# 01 - Build the city list

Cleans Census population estimates, selects the card cities (top 100 by population, every state capital, and the top 2 cities per state), geocodes them, pulls historical temperatures from Open-Meteo, and writes `data/processed/city_list_set_1.csv`.

**Run from the `notebooks/` folder.**

## 1. Load raw data

### Generation of the cities for Set 1 of the City Trading Cards

A complete code that does all the data cleaning and processing to generate the final spreadsheet with all the data for the cities in our first set of city trading cards.

Steps include:
- Importing US census data
- Importing US capital city data
- Importing Hawaii census data
- Combining all city data
- Removing Census Designated Places (CDPs) that are not cities
- Remove duplicate cities
- Rank all cities (overall, by state)
- Remove cities that are not one of the following: in the top 100 of the most populous cities in the USA, in the top 2 of the most populous cities in the state, capital of the state
- Geocode the filtered cities to get latitude and longitude
- Calculate the average monthly temperatures of each city

Final result is a spreadsheet with the following columns:
- City name
- State name
- State ID
- Capital
- Population (2023 Estimate)
- Overall Rank
- State Rank
- Latitude
- Longitude
- Jan
- Feb
- Mar
- Apr
- May
- Jun
- Jul
- Aug
- Sep
- Oct
- Nov
- Dec

## 2. Clean Census data and select cities

#### Population Data Files

US Census Data:
- Source: https://www.census.gov/data/tables/time-series/demo/popest/2020s-total-cities-and-towns.html
- Filename: SUB-IP-EST2023-ANNRNK.xlsx
- File formatting required manual copy and pasting of data into a .csv (us_census_city_pop.csv)

US Capital Data:
- Source:
- Filename: us_capitals.csv

Hawaii Data:
- Source:
- Filename: hawaii_over20k.csv
- Removed East Honolulu from data file

## 3. Geocode (Nominatim / OpenStreetMap)

#### Import Data

## 4. Monthly average temperatures (Open-Meteo, 2020-2024)

In [ ]:
import pandas as pd

census_df_path = '../data/raw/us_census_city_pop.csv'
census_df = pd.read_csv(census_df_path)

capitals_df_path = '../data/raw/us_capitals.csv'
capitals_df = pd.read_csv(capitals_df_path)

hawaii_df_path = '../data/raw/hawaii_over20k.csv'
hawaii_df = pd.read_csv(hawaii_df_path)

## 5. Round values

#### Clean Data

## 6. Export

In [ ]:
from us import states

# Find the index of the 'Geographic Area' column
geographic_area_index = census_df.columns.get_loc('Geographic Area')

# Insert the new columns 'City' and 'State' after the 'Geographic Area' column
census_df.insert(geographic_area_index + 1, 'City', '')
census_df.insert(geographic_area_index + 2, 'State', '')

# Split the 'Geographic Area' column at the comma and space
census_df[['City', 'State']] = census_df['Geographic Area'].str.split(', ', expand=True)

# Drop unused columns
census_df = census_df.drop('Geographic Area', axis=1)
census_df = census_df.drop('2020 Baseline', axis=1)
census_df = census_df.drop('2020 Est', axis=1)
census_df = census_df.drop('2021 Est', axis=1)
census_df = census_df.drop('2022 Est', axis=1)

# Clean city names
census_df['City'] = census_df['City'].str.replace(' city', '', case=True)
census_df['City'] = census_df['City'].str.replace(' (balance)', '', case=True)
census_df['City'] = census_df['City'].str.replace(' municipality', '', case=True)
census_df['City'] = census_df['City'].str.replace(' town', '', case=True)
census_df['City'] = census_df['City'].str.replace(' village', '', case=True)
census_df['City'] = census_df['City'].str.replace(' borough', '', case=True)
census_df['City'] = census_df['City'].str.replace(' and borough', '', case=True)
census_df['City'] = census_df['City'].str.replace(' township', '', case=True)
census_df['City'] = census_df['City'].replace('Urban Honolulu CDP', 'Honolulu')
census_df['City'] = census_df['City'].replace('Nashville-Davidson metropolitan government', 'Nashville')
census_df['City'] = census_df['City'].replace('Louisville/Jefferson County metro government', 'Louisville')
census_df['City'] = census_df['City'].replace('Lexington-Fayette urban county', 'Lexington')

# Find the index of the 'State' column
state_index = census_df.columns.get_loc('State')

# Add columns
census_df.insert(state_index + 1, 'State ID', '')
census_df.insert(state_index + 2, 'Capital', '')
census_df.insert(state_index + 4, 'Overall Rank', '')
census_df.insert(state_index + 5, 'State Rank', '')
census_df.insert(state_index + 6, 'Latitude', '')
census_df.insert(state_index + 7, 'Longitude', '')
census_df.insert(state_index + 8, 'Jan', '')
census_df.insert(state_index + 9, 'Feb', '')
census_df.insert(state_index + 10, 'Mar', '')
census_df.insert(state_index + 11, 'Apr', '')
census_df.insert(state_index + 12, 'May', '')
census_df.insert(state_index + 13, 'Jun', '')
census_df.insert(state_index + 14, 'Jul', '')
census_df.insert(state_index + 15, 'Aug', '')
census_df.insert(state_index + 16, 'Sep', '')
census_df.insert(state_index + 17, 'Oct', '')
census_df.insert(state_index + 18, 'Nov', '')
census_df.insert(state_index + 19, 'Dec', '')

# Select the desired columns from capitals_df
capitals_subset = capitals_df[["City", "State", "State ID", "2023 Est"]].copy()

# Select the desired columns from hawaii_df
hawaii_subset = hawaii_df[["City", "State", "State ID", "2023 Est"]].copy()

# Concatenate the two dataframes
combined_df = pd.concat([capitals_subset, hawaii_subset], ignore_index=True)

# Append the rows from combined_df to census_df
census_df = pd.concat([census_df, combined_df], ignore_index=True)
census_df = census_df[census_df['City'] != 'Boise City']

# Convert population values to integers
census_df['2023 Est'] = census_df['2023 Est'].astype(str)
census_df['2023 Est'] = census_df['2023 Est'].str.replace(',', '', regex=False)
census_df['2023 Est'] = pd.to_numeric(census_df['2023 Est'], errors='coerce').astype('Int64')

# Function to convert state name to abbreviation
def get_state_abbreviation(state_name):
    if state_name == "District of Columbia":
        return "DC"
    else:
        state_obj = states.lookup(state_name)
        if state_obj:
            return state_obj.abbr
        else:
            return None

# Apply the function to the 'State' column and populate 'State ID'
census_df['State ID'] = census_df['State'].apply(get_state_abbreviation)

# Delete duplicate rows based on 'City' and 'State ID'
census_df.drop_duplicates(subset=['City', 'State ID'], keep='first', inplace=True)

# Identify and mark capital cities
census_df['Capital'] = 'NO'  # Initialize the 'Capital' column with 'NO'

# Create a set of (City, State) pairs from capitals_df for efficient lookup
capital_locations = set(zip(capitals_df['City'], capitals_df['State']))

# Iterate through census_df and update the 'Capital' column based on both City and State
for index, row in census_df.iterrows():
    if row['City'] == 'Washington' and row['State ID'] == 'DC':
        census_df.loc[index, 'Capital'] = 'YES'
    elif (row['City'], row['State']) in capital_locations:
        census_df.loc[index, 'Capital'] = 'YES'

# Sort cities by population
census_df.sort_values(by='2023 Est', ascending=False, inplace=True)
census_df['Overall Rank'] = census_df['2023 Est'].rank(method='dense', ascending=False).astype(int)
census_df['State Rank'] = census_df.groupby('State')['2023 Est'].rank(method='dense', ascending=False).astype(int)

# Assign specific 'Population Rank (by State)' values
for index, row in census_df.iterrows():
    if row['City'] == 'Augusta' and row['State'] == 'Maine':
        census_df.loc[index, 'State Rank'] = 13
    elif row['City'] == 'Pierre' and row['State'] == 'South Dakota':
        census_df.loc[index, 'State Rank'] = 9
    elif row['City'] == 'Montpelier' and row['State'] == 'Vermont':
        census_df.loc[index, 'State Rank'] = 17

# --- Filtering the data ---

# 1. Keep the 100 most populous cities
top_100 = census_df[census_df['Overall Rank'] <= 100].copy()

# 2. Keep each city that is a capital
capitals = census_df[census_df['Capital'] == 'YES'].copy()

# Combine the initial selections
intermediate_df = pd.concat([top_100, capitals], ignore_index=True).drop_duplicates(subset=['City', 'State']).copy()

# Identify states with only one city in intermediate_df
state_counts = intermediate_df['State'].value_counts()
single_city_states = state_counts[state_counts == 1].index.tolist()

# Identify cities to add from census_df
additional_cities = []
for state in single_city_states:
    cities_in_intermediate = intermediate_df[intermediate_df['State'] == state]['City'].tolist()
    # Find the largest city in census_df for the current state that is NOT in intermediate_df
    largest_remaining_city = census_df[
        (census_df['State'] == state) & (~census_df['City'].isin(cities_in_intermediate))
    ].sort_values(by='2023 Est', ascending=False).head(1)
    if not largest_remaining_city.empty:
        additional_cities.append(largest_remaining_city.iloc[0].copy())

# Convert the list of additional cities to a DataFrame
additional_cities_df = pd.DataFrame(additional_cities)

# Combine with the intermediate_df to create the final filtered_df
filtered_df = pd.concat([intermediate_df, additional_cities_df], ignore_index=True).drop_duplicates(subset=['City', 'State']).copy()

# Sort the final filtered dataframe by overall population rank
filtered_df.sort_values(by='Overall Rank', inplace=True)
#census_df.sort_values(by='Overall Rank', inplace=True)

#### Add Latitude and Longitude to each city

In [ ]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# --- Geocoding to get Latitude and Longitude ---
geolocator = Nominatim(user_agent="city_locator")

def geocode_city(row):
    city = row['City']
    state = row['State']
    try:
        location = geolocator.geocode(f"{city}, {state}", timeout=10)
        if location:
            return pd.Series({'Latitude': location.latitude, 'Longitude': location.longitude})
        else:
            return pd.Series({'Latitude': None, 'Longitude': None})
    except GeocoderTimedOut:
        print(f"Timeout occurred for {city}, {state}. Retrying...")
        return geocode_city(row)
    except GeocoderServiceError as e:
        print(f"Geocoding service error for {city}, {state}: {e}")
        return pd.Series({'Latitude': None, 'Longitude': None})
    except Exception as e:
        print(f"An error occurred for {city}, {state}: {e}")
        return pd.Series({'Latitude': None, 'Longitude': None})

# Apply the geocoding function to each row of the filtered_df
filtered_df[['Latitude', 'Longitude']] = filtered_df.apply(geocode_city, axis=1)

#### Calculate Average Monthly Temperatures for each city

In [ ]:
import requests
import json
from datetime import datetime
import time
import random

# --- Function to fetch historical weather data with delay ---
def get_monthly_temps(latitude, longitude, city, state, delay=5, max_retries=5, initial_retry_wait=60): # Increased initial wait
    print(f"Fetching temperature data for: {city}, {state}")
    base_url = "https://archive-api.open-meteo.com/v1/archive"
    start_date = "2020-01-01"
    end_date = "2024-12-31"
    monthly_temps = {}
    all_daily_temps = []
    retries = 0
    retry_wait = initial_retry_wait

    while retries < max_retries:
        try:
            params = {
                "latitude": latitude,
                "longitude": longitude,
                "start_date": start_date,
                "end_date": end_date,
                "daily": "temperature_2m_mean",
                "timezone": "America/New_York"  # Adjust timezone if needed
            }
            response = requests.get(base_url, params=params)
            response.raise_for_status()  # Raise an exception for bad status codes
            data = response.json()

            if 'daily' in data and 'time' in data['daily'] and 'temperature_2m_mean' in data['daily']:
                times = data['daily']['time']
                temperatures_celsius = data['daily']['temperature_2m_mean']
                for i in range(len(times)):
                    date = datetime.strptime(times[i], '%Y-%m-%d')
                    month = date.month
                    year = date.year
                    fahrenheit = (temperatures_celsius[i] * 9/5) + 32
                    all_daily_temps.append({'month': month, 'year': year, 'temp_f': fahrenheit})

                monthly_avg = pd.DataFrame(all_daily_temps).groupby('month')['temp_f'].mean().to_dict()
                return {month: avg for month, avg in monthly_avg.items()}
            else:
                print(f"Weather data not found for {city}, {state}")
                return None

        except requests.exceptions.RequestException as e:
            if response is not None and response.status_code == 429:
                print(f"Rate limit exceeded for {city}, {state}. Waiting {retry_wait} seconds before retrying...")
                time.sleep(retry_wait)
                retries += 1
                retry_wait *= 2  # Exponentially increase wait time
            else:
                print(f"Error fetching weather data for {city}, {state}: {e}")
                return None
        except json.JSONDecodeError:
            print(f"Error decoding JSON response for {city}, {state}")
            return None
        except Exception as e:
            print(f"An error occurred for {city}, {state}: {e}")
            return None
        finally:
            time.sleep(delay)  # Base delay between all requests

    print(f"Max retries reached for {city}, {state}. Unable to fetch weather data.")
    return None

# --- Apply the function to each city in filtered_df ---
for index, row in filtered_df.iterrows():
    latitude = row['Latitude']
    longitude = row['Longitude']
    city = row['City']
    state = row['State']

    if pd.notna(latitude) and pd.notna(longitude):
        monthly_temperatures = get_monthly_temps(latitude, longitude, city, state)
        if monthly_temperatures:
            filtered_df.loc[index, 'Jan'] = monthly_temperatures.get(1)
            filtered_df.loc[index, 'Feb'] = monthly_temperatures.get(2)
            filtered_df.loc[index, 'Mar'] = monthly_temperatures.get(3)
            filtered_df.loc[index, 'Apr'] = monthly_temperatures.get(4)
            filtered_df.loc[index, 'May'] = monthly_temperatures.get(5)
            filtered_df.loc[index, 'Jun'] = monthly_temperatures.get(6)
            filtered_df.loc[index, 'Jul'] = monthly_temperatures.get(7)
            filtered_df.loc[index, 'Aug'] = monthly_temperatures.get(8)
            filtered_df.loc[index, 'Sep'] = monthly_temperatures.get(9)
            filtered_df.loc[index, 'Oct'] = monthly_temperatures.get(10)
            filtered_df.loc[index, 'Nov'] = monthly_temperatures.get(11)
            filtered_df.loc[index, 'Dec'] = monthly_temperatures.get(12)
    else:
        print(f"Latitude or Longitude missing for {city}, {state}. Skipping temperature data.")

#### Final Rounding

In [ ]:
# --- Rounding the values ---
filtered_df['Latitude'] = filtered_df['Latitude'].round(3)
filtered_df['Longitude'] = filtered_df['Longitude'].round(3)

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

for month in months:
    # 1. Ensure the column is treated as numeric first
    filtered_df[month] = pd.to_numeric(filtered_df[month], errors='coerce')

    # 2. Round the numeric values
    filtered_df[month] = filtered_df[month].round(0)

    # 3. Convert to the nullable integer type
    filtered_df[month] = filtered_df[month].astype('Int64')

#### Export to .csv

In [ ]:
export_filename = 'city_list_set_1.csv'
export_path = f"../data/processed/{export_filename}"
filtered_df.to_csv(export_path, index=False)